# KGA: Knowledge (Graph) Fusion by Bidirectional Information Aggregation (essentially "rewiring" attention) 
This teaching notebook illustrates **all key steps** of a KGA-style pipeline.
We use a tiny knowledge graph and also a very tiny LLM but illustrate all
the key steps, as a foundationa for doing it it scale, namely:

1) represent knowledge as triples
2) score triples vs question
3) show token→triple and triple→token relevance maps
4) build a per-layer KV memory from selected triples
5) patch Llama attention to prepend memory KV and rewire attention
6) print a small attention slice to show the rewiring
7) compare baseline vs prompt-stuffing vs KGA-fusion outputs

- This is an implementation of the approach for neural knowledge fusion put forth in
    - Zhai, S., Qi, G., Wang, Y., & Meng, Y. (2025). <a href="https://arxiv.org/abs/2507.08704">Knowledge Fusion via Bidirectional Information Aggregation.</a> arXiv preprint arXiv:2507.08704.
- The slides for Lecture 11: Neurosymbolic lowering methods: inference-time, deep integration of knowledge graphs with LLMs (datasciencey.github.io/datasci290-neurosymbolic-ai) may be helpful in understanding the approach (paper)
- Implementation notes:
    - Keep `model_id` **Llama-family** for the attention patch.
    - Decoding is greedy; caching is disabled for stability with patched attention.


In [ ]:
#Conda environment setup, and Jupyter Notebook configuration

!conda create -n kga-demo python=3.11 -y
#!conda activate kga-demo
#!conda install -y numpy=1.26
#!conda install -y pytorch torchvision torchaudio -c pytorch
#!pip install -U transformers accelerate sentencepiece tokenizers safetensors
#!conda activate kga-demo
#!conda install -y ipykernel jupyter

#python -m ipykernel install --user --name kga-demo --display-name "Python (kga-demo)"

In [1]:
# 0) Imports + deterministic-ish setup
import os, math, types, random, re
import torch
from dataclasses import dataclass
from typing import List, Tuple, Dict
from transformers import AutoTokenizer, AutoModelForCausalLM
import inspect
import torch.nn as nn

seed = 42
random.seed(seed)
torch.manual_seed(seed)

def pick_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = pick_device()
print("device:", device)


ModuleNotFoundError: No module named 'torch'

In [3]:
# 1) Load a small, non-gated HF model (Llama-family)
# Swap ONLY this model_id to try other HF models.
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

# Ensure pad exists (HF generate needs this to behave)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if (device.type == "cuda") else torch.float32,
).to(device)

model.eval()

# Also set these on generation_config (some models rely on it)
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id

print("model:", model_id)



`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model: TinyLlama/TinyLlama-1.1B-Chat-v1.0


## 2) Knowledge as Triples

In [4]:
Trip = Tuple[str, str, str]

knowledge_triples: List[Trip] = [
    ("ICML", "held_in", "Dubai"),
    ("Dubai", "located_in", "United Arab Emirates"),
    ("WWW_2025", "held_in", "Sydney"),
    ("Sydney", "located_in", "Australia"),
    ("Paris", "capital_of", "France"),
]

def triple_to_text(t: Trip) -> str:
    h, r, o = t
    return f"( {h} , {r} , {o} )"

triple_texts = [triple_to_text(t) for t in knowledge_triples]
triple_texts


['( ICML , held_in , Dubai )',
 '( Dubai , located_in , United Arab Emirates )',
 '( WWW_2025 , held_in , Sydney )',
 '( Sydney , located_in , Australia )',
 '( Paris , capital_of , France )']

## 3) Triple scoring

In [5]:
def _normalize(s: str) -> List[str]:
    s = s.lower()
    s = re.sub(r"[^a-z0-9_]+", " ", s)
    toks = [t for t in s.split() if t]
    return toks

def score_triple(question: str, t: Trip) -> float:
    q = set(_normalize(question))
    tt = set(_normalize(triple_to_text(t)))
    if not q or not tt:
        return 0.0
    return len(q & tt) / len(q | tt)

def rank_triples(question: str, triples: List[Trip]):
    scored = [(score_triple(question, t), t) for t in triples]
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored

question = "In which country is NEOCONF_2026 held? Answer with only the country."
#question = "In which country is WWW_2025 held? Answer with only the country."
ranked = rank_triples(question, knowledge_triples)
ranked


[(0.0, ('ICML', 'held_in', 'Dubai')),
 (0.0, ('Dubai', 'located_in', 'United Arab Emirates')),
 (0.0, ('WWW_2025', 'held_in', 'Sydney')),
 (0.0, ('Sydney', 'located_in', 'Australia')),
 (0.0, ('Paris', 'capital_of', 'France'))]

## 4) Token ↔ Triple relevance maps

In [6]:
q_tokens = _normalize(question)
triple_tokens = [set(_normalize(triple_to_text(t))) for t in knowledge_triples]

token_to_triples: Dict[str, List[int]] = {}
for tok in q_tokens:
    token_to_triples[tok] = [j for j, ts in enumerate(triple_tokens) if tok in ts]

triple_to_tokens: Dict[int, List[str]] = {}
for j, ts in enumerate(triple_tokens):
    triple_to_tokens[j] = [tok for tok in q_tokens if tok in ts]

print("Question tokens:", q_tokens)
print("\nToken → triples (indices):")
for tok, hits in token_to_triples.items():
    if hits:
        print(f"  {tok:>12}: {hits}")

print("\nTriple → supporting tokens:")
for j, hits in triple_to_tokens.items():
    if hits:
        print(f"  {j:>2} {triple_to_text(knowledge_triples[j])}  <-- {hits}")


Question tokens: ['in', 'which', 'country', 'is', 'neoconf_2026', 'held', 'answer', 'with', 'only', 'the', 'country']

Token → triples (indices):

Triple → supporting tokens:


## 5) Select top-k triples + weights

In [7]:
def select_topk_triples(question: str, triples: List[Trip], k: int = 2):
    ranked = rank_triples(question, triples)
    selected = [t for s, t in ranked[:k]]
    scores = torch.tensor([max(s, 1e-6) for s, _ in ranked[:k]], dtype=torch.float32)
    weights = scores / scores.sum()
    return selected, weights

selected, weights = select_topk_triples(question, knowledge_triples, k=2)
selected_texts = [triple_to_text(t) for t in selected]

print("Selected:")
for t, w in zip(selected, weights):
    print(f"  w={float(w):.3f} {t}")

selected_texts


Selected:
  w=0.500 ('ICML', 'held_in', 'Dubai')
  w=0.500 ('Dubai', 'located_in', 'United Arab Emirates')


['( ICML , held_in , Dubai )', '( Dubai , located_in , United Arab Emirates )']

## 6) Build per-layer KV memory from selected triples

In [8]:
@dataclass
class TripleMemory:
    kv_per_layer: List[Tuple[torch.Tensor, torch.Tensor]]  # (K,V) per layer: [1, kvh, M, D]
    valid_mask: torch.Tensor                               # [1, M] bool

def _encode_single(text: str, max_tokens: int = 32):
    enc = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=max_tokens,
        padding="max_length"
    )
    return enc["input_ids"].to(device), enc["attention_mask"].to(device)

@torch.no_grad()
def build_triple_memory(triple_texts: List[str], weights: torch.Tensor, max_tokens_per_triple: int = 24) -> TripleMemory:
    encs = [_encode_single(t, max_tokens=max_tokens_per_triple) for t in triple_texts]
    B = len(encs)
    w = weights.to(device).view(B, 1, 1, 1)

    all_hidden = []
    all_valid = []
    for (ids, am) in encs:
        out = model.model(
            input_ids=ids,
            attention_mask=am,
            output_hidden_states=True,
            use_cache=False,
            return_dict=True,
        )
        all_hidden.append(out.hidden_states)      # tuple length L+1, each [1,T,H]
        all_valid.append(am[0].bool())            # [T]

    layers = model.model.layers
    num_layers = len(layers)

    valid_mask = torch.cat(all_valid, dim=0).view(1, -1)  # [1, B*T]
    kv_per_layer = []

    for layer_idx in range(num_layers):
        attn = layers[layer_idx].self_attn
        kv_heads = getattr(attn, "num_key_value_heads", None)
        if kv_heads is None:
            kv_heads = getattr(model.config, "num_key_value_heads", getattr(model.config, "num_attention_heads"))
        head_dim = attn.k_proj.out_features // kv_heads

        ks, vs = [], []
        for j in range(B):
            h = all_hidden[j][layer_idx]               # [1,T,H] proxy
            am = all_valid[j].view(1, -1, 1)           # [1,T,1]

            k = attn.k_proj(h).view(1, -1, kv_heads, head_dim).transpose(1, 2)  # [1,kvh,T,D]
            v = attn.v_proj(h).view(1, -1, kv_heads, head_dim).transpose(1, 2)

            k = k * am[:, None, :, :]
            v = v * am[:, None, :, :]
            v = v * w[j]

            ks.append(k)
            vs.append(v)

        K = torch.cat(ks, dim=2).contiguous()  # [1,kvh,M,D]
        V = torch.cat(vs, dim=2).contiguous()
        kv_per_layer.append((K, V))

    return TripleMemory(kv_per_layer=kv_per_layer, valid_mask=valid_mask)

triple_memory = build_triple_memory(selected_texts, weights, max_tokens_per_triple=24)
print("layers:", len(triple_memory.kv_per_layer))
print("layer0 K:", tuple(triple_memory.kv_per_layer[0][0].shape))
print("valid_mask:", tuple(triple_memory.valid_mask.shape), "valid:", int(triple_memory.valid_mask.sum()))


layers: 22
layer0 K: (1, 4, 48, 64)
valid_mask: (1, 48) valid: 27


## 7) Patch attention to prepend memory KV
### (this is THE key step - the "attention rewiring")

In [9]:
import math, types
import torch
import torch.nn as nn

def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
    # [B, kvh, T, D] -> [B, kvh*n_rep, T, D]
    if n_rep == 1:
        return hidden_states
    B, kvh, T, D = hidden_states.shape
    hidden_states = hidden_states[:, :, None, :, :].expand(B, kvh, n_rep, T, D)
    return hidden_states.reshape(B, kvh * n_rep, T, D)

def _ensure_4d_additive_mask(attention_mask, dtype, tgt_len, device):
    """
    Normalize attention_mask into additive 4D mask: [B, 1, tgt_len, src_len]
    Accepts:
      - None
      - 2D mask [B, src_len] where 1=keep, 0=mask
      - 4D additive mask [B, 1, tgt_len, src_len]
    """
    if attention_mask is None:
        return None

    if attention_mask.dim() == 4:
        return attention_mask.to(dtype=dtype, device=device)

    if attention_mask.dim() == 2:
        neg_inf = torch.finfo(dtype).min
        keep = attention_mask.to(device=device).bool()  # True means keep
        add = torch.where(
            keep,
            torch.zeros_like(keep, dtype=dtype, device=device),
            torch.full_like(keep, neg_inf, dtype=dtype, device=device),
        )
        # [B, 1, tgt_len, src_len]
        return add[:, None, None, :].expand(add.size(0), 1, tgt_len, add.size(1))

    raise ValueError(f"Unsupported attention_mask shape: {attention_mask.shape}")

def _make_mem_additive_mask(valid_mask_1xM: torch.Tensor, dtype: torch.dtype, tgt_len: int, bsz: int, device):
    """
    valid_mask_1xM: [1, M] bool; True means "valid memory token".
    Returns [B, 1, tgt_len, M] additive mask.
    """
    neg_inf = torch.finfo(dtype).min
    vm = valid_mask_1xM.to(device=device).bool()  # [1, M]
    add = torch.where(
        vm,
        torch.zeros_like(vm, dtype=dtype, device=device),
        torch.full_like(vm, neg_inf, dtype=dtype, device=device),
    )
    return add.view(1, 1, 1, -1).expand(bsz, 1, tgt_len, -1)

class KGAWrappedAttention(nn.Module):
    """
    Module-swap wrapper:
      - Baseline: delegate to original HF attention
      - KGA enabled: do KV-prefix injection + attention in this wrapper
    Returns EXACTLY 2 outputs: (attn_output, attn_probs_or_None)
    """
    def __init__(self, base_attn: nn.Module, triple_memory, layer_idx: int, model_config, apply_rotary_pos_emb):
        super().__init__()
        self.base_attn = base_attn
        self.kga_triple_memory = triple_memory
        self.kga_layer_idx = layer_idx

        # toggles used by your existing helper funcs
        self.kga_enabled = False
        self.kga_capture = False
        self.kga_last_attn = None

        self._cfg = model_config
        self._apply_rope = apply_rotary_pos_emb

        # keep cache-related attrs that HF may look for
        if hasattr(base_attn, "layer_idx"):
            self.layer_idx = base_attn.layer_idx
        else:
            self.layer_idx = layer_idx

    def _return_two(self, out):
        """
        HF attention variants sometimes return (o, attn) or (o, attn, pkv).
        LlamaDecoderLayer in your stacktrace expects TWO values to unpack.
        """
        if isinstance(out, tuple) or isinstance(out, list):
            if len(out) >= 2:
                return out[0], out[1]
            if len(out) == 1:
                return out[0], None
        return out, None

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        position_ids=None,
        past_key_value=None,
        output_attentions=False,
        use_cache=False,
        cache_position=None,
        position_embeddings=None,
        **kwargs
    ):
        # ---- BASELINE PATH (no KGA): call original attention exactly ----
        if not getattr(self, "kga_enabled", False):
            out = self.base_attn(
                hidden_states=hidden_states,
                attention_mask=attention_mask,
                position_ids=position_ids,
                past_key_value=past_key_value,
                output_attentions=output_attentions,
                use_cache=use_cache,
                cache_position=cache_position,
                position_embeddings=position_embeddings,
                **kwargs
            )
            return self._return_two(out)

        # ---- KGA PATH: do KV-prefix injection ourselves ----
        device = hidden_states.device
        bsz, q_len, _ = hidden_states.size()
        attn = self.base_attn  # shorthand

        # projections
        query_states = attn.q_proj(hidden_states)
        key_states   = attn.k_proj(hidden_states)
        value_states = attn.v_proj(hidden_states)

        hidden_size = query_states.shape[-1]
        num_heads = getattr(self._cfg, "num_attention_heads")
        head_dim  = hidden_size // num_heads
        kv_heads  = getattr(self._cfg, "num_key_value_heads", num_heads)
        kv_groups = num_heads // kv_heads

        # [B,h,T,D] / [B,kvh,T,D]
        query_states = query_states.view(bsz, q_len, num_heads, head_dim).transpose(1, 2)
        key_states   = key_states.view(bsz, q_len, kv_heads,  head_dim).transpose(1, 2)
        value_states = value_states.view(bsz, q_len, kv_heads, head_dim).transpose(1, 2)

        # rotary
        if position_embeddings is not None:
            cos, sin = position_embeddings
        else:
            # common modern llama path
            cos, sin = attn.rotary_emb(value_states, position_ids=position_ids)

        try:
            query_states, key_states = self._apply_rope(query_states, key_states, cos, sin, position_ids)
        except TypeError:
            query_states, key_states = self._apply_rope(query_states, key_states, cos, sin)

        # prepend triple-memory KV (per layer)
        mem_k, mem_v = self.kga_triple_memory.kv_per_layer[self.kga_layer_idx]  # [1,kvh,M,D]
        mem_k = mem_k.to(device=device, dtype=key_states.dtype).expand(bsz, -1, -1, -1).contiguous()
        mem_v = mem_v.to(device=device, dtype=value_states.dtype).expand(bsz, -1, -1, -1).contiguous()

        key_states   = torch.cat([mem_k, key_states], dim=2)   # src_len += M
        value_states = torch.cat([mem_v, value_states], dim=2)

        # mask: extend for memory tokens
        attn_mask_4d = _ensure_4d_additive_mask(attention_mask, dtype=query_states.dtype, tgt_len=q_len, device=device)
        if attn_mask_4d is not None:
            mem_add = _make_mem_additive_mask(self.kga_triple_memory.valid_mask, attn_mask_4d.dtype, q_len, bsz, device)
            attention_mask = torch.cat([mem_add, attn_mask_4d], dim=-1)  # concat on src_len
        else:
            attention_mask = None

        # expand kv for GQA/MQA
        key_states_full   = repeat_kv(key_states,   kv_groups)  # [B,h,src,D]
        value_states_full = repeat_kv(value_states, kv_groups)

        # attention
        attn_weights = torch.matmul(query_states, key_states_full.transpose(2, 3)) / math.sqrt(head_dim)
        if attention_mask is not None:
            attn_weights = attn_weights + attention_mask

        attn_probs  = torch.softmax(attn_weights.float(), dim=-1).to(attn_weights.dtype)
        attn_output = torch.matmul(attn_probs, value_states_full)

        # optional capture
        if getattr(self, "kga_capture", False):
            # first batch, head0, query0, first 12 keys
            self.kga_last_attn = attn_probs[0, 0, 0, :12].detach().float().cpu()

        # back to [B,T,H]
        attn_output = attn_output.transpose(1, 2).contiguous().view(bsz, q_len, num_heads * head_dim)
        attn_output = attn.o_proj(attn_output)

        # IMPORTANT: return exactly 2 values
        if output_attentions:
            return attn_output, attn_probs
        return attn_output, None


def patch_llama_attention(model, triple_memory):
    """
    Patch: swap layer.self_attn modules with KGAWrappedAttention wrappers.

    Returns: list of wrapper modules in layer order (so your set_kga_layer_window works).
    """
    from transformers.models.llama import modeling_llama
    LlamaAttention = modeling_llama.LlamaAttention
    apply_rotary_pos_emb = modeling_llama.apply_rotary_pos_emb

    # Make sure we are in inference mode (avoids weird slowdowns / checkpointing surprises)
    model.eval()
    if hasattr(model, "gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()

    layers = model.model.layers
    wrappers = []

    # If already patched, unpatch first (so re-running cell is safe)
    if hasattr(model, "_kga_original_self_attn"):
        for i, layer in enumerate(layers):
            if i < len(model._kga_original_self_attn):
                layer.self_attn = model._kga_original_self_attn[i]
        delattr(model, "_kga_original_self_attn")

    originals = []
    for i, layer in enumerate(layers):
        base = layer.self_attn
        if not isinstance(base, LlamaAttention):
            raise RuntimeError(
                f"Layer {i} self_attn is not LlamaAttention (got {type(base)}). "
                "Use a Llama-family model_type."
            )
        originals.append(base)

        wrapped = KGAWrappedAttention(
            base_attn=base,
            triple_memory=triple_memory,
            layer_idx=i,
            model_config=model.config,
            apply_rotary_pos_emb=apply_rotary_pos_emb,
        )

        layer.self_attn = wrapped
        wrappers.append(wrapped)

    model._kga_original_self_attn = originals
    print(f"Patched {len(wrappers)} layers by swapping self_attn modules (baseline delegates to original).")
    return wrappers


def unpatch_llama_attention(model):
    """Restore original HF attention modules (full fallback)."""
    if not hasattr(model, "_kga_original_self_attn"):
        print("Model is not patched (nothing to unpatch).")
        return
    layers = model.model.layers
    originals = model._kga_original_self_attn
    for i, layer in enumerate(layers):
        if i < len(originals):
            layer.self_attn = originals[i]
    delattr(model, "_kga_original_self_attn")
    print("Restored original self_attn modules.")


In [10]:
# Re-run patch after redefining
attn_modules = patch_llama_attention(model, triple_memory)

Patched 22 layers by swapping self_attn modules (baseline delegates to original).


## 8) Generation comparison + attention glimpse

In [11]:
def set_kga_layer_window(attn_modules, start: int, end: int):
    for i, m in enumerate(attn_modules):
        m.kga_enabled = (start <= i < end)

def set_kga_capture(attn_modules, enabled: bool):
    for m in attn_modules:
        m.kga_capture = enabled
        m.kga_last_attn = None

def print_attention_glimpse(attn_modules):
    for i, m in enumerate(attn_modules):
        if m.kga_last_attn is not None:
            print(f"Captured attention slice from layer {i}:")
            print(m.kga_last_attn.numpy())
            return
    print("No attention captured.")


SYSTEM_PROMPT = "You are a helpful assistant. Answer concisely and directly."

@torch.no_grad()
def _encode_prompt(prompt: str, max_prompt_tokens: int = 512):
    """
    Robust prompt encoding:
    - If the tokenizer has a chat template, use it (TinyLlama-Chat expects this).
    - Otherwise fall back to plain tokenization.
    Returns (input_ids, attention_mask) on the correct device.
    """
    if getattr(tokenizer, "chat_template", None):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]
        # IMPORTANT: return_dict=True so we reliably get tensors (no BatchEncoding confusion)
        enc = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
            truncation=True,
            max_length=max_prompt_tokens,
        )
        input_ids = enc["input_ids"].to(device)
        attention_mask = enc.get("attention_mask", None)
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids, device=device)
        else:
            attention_mask = attention_mask.to(device)
        return input_ids, attention_mask

    # Non-chat fallback
    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_prompt_tokens,
    )
    return enc["input_ids"].to(device), enc["attention_mask"].to(device)


def _clean_generation(text: str) -> str:
    # Common junk tokens / wrappers that show up with Llama chat models
    text = text.strip()

    # strip repeated or standalone end tokens
    for bad in ["</s>", "<s>"]:
        if text == bad:
            text = ""
        text = text.replace(bad, "").strip()

    # strip common “formatting attractors”
    if text in ['"""', "```", "''"]:
        text = ""

    # if it starts with triple quotes but nothing useful follows, drop it
    if text.startswith('"""') and len(text) <= 6:
        text = ""

    return text.strip()


@torch.no_grad()
def generate_text(
    prompt: str,
    max_new_tokens: int = 40,
    temperature: float = 0.0,
    top_p: float = 0.95,
    repetition_penalty: float = 1.10,
    no_repeat_ngram_size: int = 3,
    min_new_tokens: int = 6,          # <-- NEW: prevents immediate </s> bail-out
    debug: bool = True
) -> str:
    input_ids, attention_mask = _encode_prompt(prompt)

    do_sample = bool(temperature and temperature > 0)

    gen_kwargs = dict(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=int(max_new_tokens),
        min_new_tokens=int(min_new_tokens),   # <-- NEW
        do_sample=do_sample,
        temperature=float(temperature) if do_sample else None,
        top_p=float(top_p) if do_sample else None,
        repetition_penalty=float(repetition_penalty),
        no_repeat_ngram_size=int(no_repeat_ngram_size),
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=False,
    )

    out = model.generate(**gen_kwargs)

    prompt_len = input_ids.shape[-1]
    gen_ids = out[0, prompt_len:]

    if debug:
        print(f"prompt_len: {prompt_len} total_len: {out.shape[-1]} gen_len: {gen_ids.shape[-1]}")
        if gen_ids.numel() > 0:
            first_id = int(gen_ids[0].item())
            print(f"first_generated_id: {first_id} is_eos: {first_id == tokenizer.eos_token_id}")

    # Decode continuation only
    text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    text = _clean_generation(text)

    # --- Auto-retry if we got junk (very common with small chat models) ---
    if text == "":
        # Retry with a tiny bit of sampling to escape degenerate “stop immediately”
        gen_kwargs["do_sample"] = True
        gen_kwargs["temperature"] = 0.3
        gen_kwargs["top_p"] = 0.95
        gen_kwargs["min_new_tokens"] = max(10, int(min_new_tokens))

        out2 = model.generate(**gen_kwargs)
        gen_ids2 = out2[0, prompt_len:]
        text2 = tokenizer.decode(gen_ids2, skip_special_tokens=True).strip()
        text2 = _clean_generation(text2)

        # Use retry result if it’s better
        if text2 != "":
            text = text2

    return text

In [12]:

def prompt_stuff(question: str, triples: List[Trip]) -> str:
    # Keep this simple: it becomes the *user message content* inside the chat template.
    lines = [
        question,
        "",
        "Knowledge (facts you may use):",
        *[triple_to_text(t) for t in triples],
        "",
        "Answer:"
    ]
    return "\n".join(lines)

In [13]:

question = "Where was WWW_2025 held_in ?"

print("\n--- Baseline (no knowledge) ---")
set_kga_layer_window(attn_modules, 0, 0)   # KGA off
set_kga_capture(attn_modules, False)
print(generate_text(question))

print("\n--- Prompt stuffing (triples as text) ---")
print(generate_text(prompt_stuff(question, selected)))

print("\n--- KGA fusion (rewired attention) ---")
mid_start = len(attn_modules)//3
mid_end = 2*len(attn_modules)//3
set_kga_layer_window(attn_modules, mid_start, mid_end)
set_kga_capture(attn_modules, True)
print(generate_text(question))

print("\nAttention glimpse (head0/query0 → first 12 keys):")
print_attention_glimpse(attn_modules)


--- Baseline (no knowledge) ---
prompt_len: 52 total_len: 92 gen_len: 40
first_generated_id: 29956 is_eos: False
WWW (World Wide Web) 2015 is the name of the conference that took place in 23-27 October 215 at the University of California

--- Prompt stuffing (triples as text) ---
prompt_len: 95 total_len: 123 gen_len: 28
first_generated_id: 29956 is_eos: False
WWW-2015 was held in Dubai, located in the United Arabic Emirate of Dubai.

--- KGA fusion (rewired attention) ---
prompt_len: 52 total_len: 64 gen_len: 12
first_generated_id: 29896 is_eos: False
1000101 of

Attention glimpse (head0/query0 → first 12 keys):
Captured attention slice from layer 7:
[9.99442378e-08 1.12906005e-02 1.10769989e-02 1.12338495e-02
 1.11796102e-02 1.15388315e-02 1.08243432e-02 1.13362977e-02
 1.13481050e-02 1.13799535e-02 1.09558580e-02 1.17564648e-02]


## Important

This notebook illustrates the *core mechanic of knowledge fusion into LLM neural networks*: we **rewire attention** so a layer *can* prepend “knowledge memory” (K,V) alongside normal prompt tokens. What it **does not** illustrate is the engineering needed for a robust, scalable KGA system.
This is on the following counts.


- **Triple retrieval + ranking (real selection)** : We took very few hand-picked triples. At scale you would retrieve many candidates, score them for the question, keep top-K, drop distractors.

- **Structured memory that preserves “triple boundaries”**: The triples are just short text tokens. At scale we would keep head/relation/tail segments distinct, track which memory positions belong to which triple etc.

- **Masking policy for memory**: The memory is simply “allowed” to be attended to. Ideally we should have explicit masking choices (which query tokens can attend to memory; per-triple gating; prevent unintended interactions). Also tune which layers get memory (early/mid/late), how many layers, and possibly different layers for different triple types.

- **How much influence memory has (calibration)**: Currently the memory competes with prompt tokens with minimal control. Ideally we require calibrated mixing/weights (so memory helps without causing instability - the random output you see for KGA fusion in this demo)

- **Robustness across models**: This is (just) a one Llama-family attention implementation ! Differenet models/variants would have different  attention implementations,  tokenization/chat formatting etc.
